In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score
)
import matplotlib.pyplot as plt
import re
from scipy.stats import chi2_contingency

In [ ]:
# load in the data
train_df = pd.read_parquet("../train_final_v3.parquet")
val_df = pd.read_parquet("../validationA_v3.parquet")


In [ ]:
# load in the data
#reviews_df_raw = pd.read_json("../amazon_reviews.json",lines=True)
#reviews_pickle = reviews_df_raw.to_pickle('reviews_raw.pkl')
reviews_df_raw = pd.read_pickle('../rebekah-idea-testing/reviews_raw.pkl')

In [ ]:
# check shape 
print(train_df.shape)
print(val_df.shape)

In [ ]:
# drop the target and make its own variable
X_train = train_df.drop(columns=['match'], axis=1)
X_val = val_df.drop(columns=['match'], axis=1)
y_train = train_df['match']
y_val = val_df['match']

In [ ]:
X_val

## Column Types

In [ ]:
for df in (X_train, X_val):
    df['missing_price'] = df['missing_price'].astype(int)

print(f"The shape of X_train: {X_train.shape}, X_val: {X_val.shape}")

In [ ]:
for df in (X_train, X_val):
    df.drop(['percent_positive', 'percent_negative', 'unique_reviewer_count'], axis=1, inplace=True)

In [ ]:
for df in (X_train, X_val):
    df['review_span'] = (df['max_date'] - df['min_date']).dt.days
    df.drop(['min_date', 'max_date'], axis=1, inplace=True)

In [ ]:
for df in (X_train, X_val):
    df['product_lifespan_days'] = df['product_lifespan'].dt.days
    df.drop('product_lifespan', axis=1, inplace=True)

In [ ]:
# STEP 1: Identify embedding columns separately
summary_embedding_cols = [col for col in X_train.columns if re.match(r'^embed_\d+$', col)]
review_embedding_cols = [col for col in X_train.columns if re.match(r'^embedding_\d+$', col)]

# Extract review and summary embedding subsets
X_emb_review_train = X_train[review_embedding_cols].copy()
X_emb_summary_train = X_train[summary_embedding_cols].copy()

X_emb_review_val = X_val[review_embedding_cols].copy()
X_emb_summary_val = X_val[summary_embedding_cols].copy()

# Drop all embedding columns from X_train and X_val
X_train_nonemb = X_train.drop(columns=review_embedding_cols + summary_embedding_cols)
X_val_nonemb   = X_val.drop(columns=review_embedding_cols + summary_embedding_cols)

# STEP 2: Preprocess non-embedding features
categorical_cols = ['category']  
numeric_cols = [col for col in X_train_nonemb.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ]
)

X_train_nonemb_proc = preprocessor.fit_transform(X_train_nonemb)
X_val_nonemb_proc = preprocessor.transform(X_val_nonemb)

# STEP 3: Scale review and summary embeddings separately
scaler_review = StandardScaler()
X_emb_review_train_scaled = scaler_review.fit_transform(X_emb_review_train)
X_emb_review_val_scaled = scaler_review.transform(X_emb_review_val)

scaler_summary = StandardScaler()
X_emb_summary_train_scaled = scaler_summary.fit_transform(X_emb_summary_train)
X_emb_summary_val_scaled = scaler_summary.transform(X_emb_summary_val)

# STEP 4: PCA on each embedding type
pca_review = PCA(n_components=50, svd_solver='auto', random_state=42)
X_emb_review_train_reduced = pca_review.fit_transform(X_emb_review_train_scaled)
X_emb_review_val_reduced = pca_review.transform(X_emb_review_val_scaled)

pca_summary = PCA(n_components=10, svd_solver='auto', random_state=42)
X_emb_summary_train_reduced = pca_summary.fit_transform(X_emb_summary_train_scaled)
X_emb_summary_val_reduced = pca_summary.transform(X_emb_summary_val_scaled)

# STEP 5: VIF filtering
vif_feature_names = (
    numeric_cols +
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols))
)
X_df_for_vif = pd.DataFrame(X_train_nonemb_proc, columns=vif_feature_names)

while True:
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_df_for_vif.columns
    vif_data["VIF"] = [variance_inflation_factor(X_df_for_vif.values, i)
                       for i in range(X_df_for_vif.shape[1])]
    
    if vif_data["VIF"].max() <= 10:
        break

    drop_feature = vif_data.sort_values("VIF", ascending=False).iloc[0]["feature"]
    print(f"Dropping {drop_feature} with VIF = {vif_data['VIF'].max():.2f}")
    X_df_for_vif = X_df_for_vif.drop(columns=[drop_feature])

# Align validation with VIF-filtered features
X_val_nonemb_proc_df = pd.DataFrame(X_val_nonemb_proc, columns=vif_feature_names)
X_val_nonemb_filtered = X_val_nonemb_proc_df[X_df_for_vif.columns]

# STEP 6: Final feature names
final_feature_names = (
    X_df_for_vif.columns.tolist() +
    [f"pca_review_emb_{i}" for i in range(X_emb_review_train_reduced.shape[1])] +
    [f"pca_summary_emb_{i}" for i in range(X_emb_summary_train_reduced.shape[1])]
)

# STEP 7: Final data matrices
X_train_final_df = pd.DataFrame(
    np.hstack([X_df_for_vif.values, X_emb_review_train_reduced, X_emb_summary_train_reduced]),
    columns=final_feature_names
)

X_val_final_df = pd.DataFrame(
    np.hstack([X_val_nonemb_filtered.values, X_emb_review_val_reduced, X_emb_summary_val_reduced]),
    columns=final_feature_names
)


In [ ]:
X_train_final_df1 = X_train_final_df
X_val_final_df1 = X_val_final_df

## Generate Feature Processor Class After Those Preprocessing Steps:

In [ ]:
class FeatureProcessor:
    def __init__(self, numeric_cols, categorical_cols):
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols

        self.preprocessor = None
        self.scaler_emb = None
        self.pca = None

        self.vif_keep_columns = None
        self.final_feature_names = None
        self.embedding_cols = None
        self.preproc_feature_names = None

    def _preprocess_dates_and_flags(self, df):
        df = df.copy()

        if 'max_date' in df.columns and 'min_date' in df.columns:
            df['review_span'] = (df['max_date'] - df['min_date']).dt.days
            df.drop(['min_date', 'max_date'], axis=1, inplace=True)

        if 'missing_price' in df.columns:
            df['missing_price'] = df['missing_price'].astype(int)

        if 'product_lifespan' in df.columns:
            df['product_lifespan_days'] = df['product_lifespan'].dt.days
            df.drop('product_lifespan', axis=1, inplace=True)

        drop_cols = [c for c in ['percent_positive', 'percent_negative', 'unique_reviewer_count'] if c in df.columns]
        df.drop(columns=drop_cols, errors='ignore', inplace=True)

        return df

    def fit(self, X_raw, skip_vif=False):
        X_raw = self._preprocess_dates_and_flags(X_raw)

        # Separate embeddings
        self.review_cols = [col for col in X_raw.columns if re.match(r'^embed_\d+$', col)]
        self.summary_cols = [col for col in X_raw.columns if re.match(r'^embedding_\d+$', col)]

        # Select columns for each embedding type
        X_emb_review = X_raw[self.review_cols]
        X_emb_summary = X_raw[self.summary_cols]
        X_emb = X_raw[self.review_cols + self.summary_cols]

        # Everything else
        X_nonemb = X_raw.drop(columns=self.review_cols + self.summary_cols)

        # Fit ColumnTransformer
        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), self.numeric_cols),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), self.categorical_cols),
            ]
        )
        X_nonemb_proc = self.preprocessor.fit_transform(X_nonemb)
        self.preproc_feature_names = (
            self.numeric_cols +
            list(self.preprocessor.named_transformers_['cat'].get_feature_names_out(self.categorical_cols))
        )

        # VIF filtering
        if not skip_vif:
            X_vif = pd.DataFrame(X_nonemb_proc, columns=self.preproc_feature_names)
            while True:
                vif_data = pd.DataFrame()
                vif_data["feature"] = X_vif.columns
                vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i)
                                   for i in range(X_vif.shape[1])]
                if vif_data["VIF"].max() <= 10:
                    break
                drop_feature = vif_data.sort_values("VIF", ascending=False).iloc[0]["feature"]
                print(f"Dropping {drop_feature} with VIF = {vif_data['VIF'].max():.2f}")
                X_vif = X_vif.drop(columns=[drop_feature])
            self.vif_keep_columns = X_vif.columns.tolist()
        else:
            X_nonemb_df = pd.DataFrame(X_nonemb_proc, columns=self.preproc_feature_names)
            X_vif = X_nonemb_df[self.vif_keep_columns]

        # Scale and reduce embeddings
        self.scaler_emb = StandardScaler()
        X_emb_scaled = self.scaler_emb.fit_transform(X_emb)

        self.pca = PCA(n_components=0.95, svd_solver='full', random_state=42)
        X_emb_reduced = self.pca.fit_transform(X_emb_scaled)

        # Final DataFrame
        self.final_feature_names = self.vif_keep_columns + [f'pca_emb_{i}' for i in range(X_emb_reduced.shape[1])]
        return pd.DataFrame(np.hstack([X_vif.values, X_emb_reduced]), columns=self.final_feature_names)

    def transform(self, X_raw):
        X_raw = self._preprocess_dates_and_flags(X_raw)

        # Separate embeddings
        self.embedding_cols = self.review_cols + self.summary_cols
        X_emb = X_raw[self.embedding_cols]
        X_nonemb = X_raw.drop(columns=self.embedding_cols)

        # Transform non-embedding
        X_nonemb_proc = self.preprocessor.transform(X_nonemb)
        X_nonemb_df = pd.DataFrame(X_nonemb_proc, columns=self.preproc_feature_names)
        X_vif = X_nonemb_df[self.vif_keep_columns]

        # Transform embeddings
        X_emb_scaled = self.scaler_emb.transform(X_emb)
        X_emb_reduced = self.pca.transform(X_emb_scaled)

        return pd.DataFrame(np.hstack([X_vif.values, X_emb_reduced]), columns=self.final_feature_names)

In [ ]:
class FeatureProcessor:
    def __init__(self, numeric_cols, categorical_cols):
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols

        self.preprocessor = None

        self.scaler_review = None
        self.scaler_summary = None

        self.pca_review = None
        self.pca_summary = None

        self.vif_keep_columns = None
        self.final_feature_names = None

        self.review_cols = None
        self.summary_cols = None

        self.preproc_feature_names = None

    def _preprocess_dates_and_flags(self, df):
        df = df.copy()

        if 'max_date' in df.columns and 'min_date' in df.columns:
            df['review_span'] = (df['max_date'] - df['min_date']).dt.days
            df.drop(['min_date', 'max_date'], axis=1, inplace=True)

        if 'missing_price' in df.columns:
            df['missing_price'] = df['missing_price'].astype(int)

        if 'product_lifespan' in df.columns:
            df['product_lifespan_days'] = df['product_lifespan'].dt.days
            df.drop('product_lifespan', axis=1, inplace=True)

        drop_cols = [c for c in ['percent_positive', 'percent_negative', 'unique_reviewer_count'] if c in df.columns]
        df.drop(columns=drop_cols, errors='ignore', inplace=True)

        return df

    def fit(self, X_raw, skip_vif=False):
        X_raw = self._preprocess_dates_and_flags(X_raw)

        # Separate embedding columns
        self.review_cols = [col for col in X_raw.columns if re.match(r'^embed_\d+$', col)]
        self.summary_cols = [col for col in X_raw.columns if re.match(r'^embedding_\d+$', col)]

        # Extract embeddings separately
        X_emb_review = X_raw[self.review_cols]
        X_emb_summary = X_raw[self.summary_cols]

        # Non-embedding columns
        X_nonemb = X_raw.drop(columns=self.review_cols + self.summary_cols)

        # Fit ColumnTransformer on non-embedding features
        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), self.numeric_cols),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), self.categorical_cols),
            ]
        )
        X_nonemb_proc = self.preprocessor.fit_transform(X_nonemb)
        self.preproc_feature_names = (
            self.numeric_cols +
            list(self.preprocessor.named_transformers_['cat'].get_feature_names_out(self.categorical_cols))
        )

        # VIF filtering
        if not skip_vif:
            X_vif = pd.DataFrame(X_nonemb_proc, columns=self.preproc_feature_names)
            while True:
                vif_data = pd.DataFrame()
                vif_data["feature"] = X_vif.columns
                vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i)
                                   for i in range(X_vif.shape[1])]
                if vif_data["VIF"].max() <= 10:
                    break
                drop_feature = vif_data.sort_values("VIF", ascending=False).iloc[0]["feature"]
                print(f"Dropping {drop_feature} with VIF = {vif_data['VIF'].max():.2f}")
                X_vif = X_vif.drop(columns=[drop_feature])
            self.vif_keep_columns = X_vif.columns.tolist()
        else:
            X_nonemb_df = pd.DataFrame(X_nonemb_proc, columns=self.preproc_feature_names)
            X_vif = X_nonemb_df[self.vif_keep_columns]

        # Scale embeddings separately
        self.scaler_review = StandardScaler()
        X_emb_review_scaled = self.scaler_review.fit_transform(X_emb_review)

        self.scaler_summary = StandardScaler()
        X_emb_summary_scaled = self.scaler_summary.fit_transform(X_emb_summary)

        # PCA separately
        self.pca_review = PCA(n_components=50, svd_solver='auto', random_state=42)
        X_emb_review_reduced = self.pca_review.fit_transform(X_emb_review_scaled)

        self.pca_summary = PCA(n_components=10, svd_solver='auto', random_state=42)
        X_emb_summary_reduced = self.pca_summary.fit_transform(X_emb_summary_scaled)

        # Compose final feature names with clear distinction
        self.final_feature_names = (
            self.vif_keep_columns +
            [f'pca_review_emb_{i}' for i in range(X_emb_review_reduced.shape[1])] +
            [f'pca_summary_emb_{i}' for i in range(X_emb_summary_reduced.shape[1])]
        )

        # Return combined DataFrame
        return pd.DataFrame(
            np.hstack([X_vif.values, X_emb_review_reduced, X_emb_summary_reduced]),
            columns=self.final_feature_names
        )

    def transform(self, X_raw):
        X_raw = self._preprocess_dates_and_flags(X_raw)

        # Separate embedding columns
        X_emb_review = X_raw[self.review_cols]
        X_emb_summary = X_raw[self.summary_cols]
        X_nonemb = X_raw.drop(columns=self.review_cols + self.summary_cols)

        # Transform non-embedding features
        X_nonemb_proc = self.preprocessor.transform(X_nonemb)
        X_nonemb_df = pd.DataFrame(X_nonemb_proc, columns=self.preproc_feature_names)
        X_vif = X_nonemb_df[self.vif_keep_columns]

        # Transform embeddings separately
        X_emb_review_scaled = self.scaler_review.transform(X_emb_review)
        X_emb_review_reduced = self.pca_review.transform(X_emb_review_scaled)

        X_emb_summary_scaled = self.scaler_summary.transform(X_emb_summary)
        X_emb_summary_reduced = self.pca_summary.transform(X_emb_summary_scaled)

        # Return combined DataFrame
        return pd.DataFrame(
            np.hstack([X_vif.values, X_emb_review_reduced, X_emb_summary_reduced]),
            columns=self.final_feature_names
        )


In [ ]:
# Final vif columns
final_vif_columns = X_df_for_vif.columns.tolist()

print(f"Final vif columns: {final_vif_columns}")

processor = FeatureProcessor(numeric_cols, categorical_cols)
processor.vif_keep_columns = final_vif_columns

X_train_final_df = processor.fit(train_df.drop(columns=['match'], axis=1), skip_vif=True)
X_val_final_df   = processor.transform(val_df.drop(columns=['match'], axis=1))

In [ ]:
# Save preprocessed columns:
X_train_final_df.to_pickle("X_train_final_df.pkl")
X_val_final_df.to_pickle("X_val_final_df.pkl")

In [ ]:
# Load preprocessed columns
X_train_final_df = pd.read_pickle("X_train_final_df.pkl")
X_val_final_df   = pd.read_pickle("X_val_final_df.pkl")

# Model Training

In [ ]:
class ModelEvaluator:
    def __init__(self, model, average='binary', display_labels=None):
        self.model = model
        self.average = average
        self.display_labels = display_labels

    def evaluate_classification(self, y_true, y_pred):
        prec = precision_score(y_true, y_pred, average=self.average)
        rec  = recall_score   (y_true, y_pred, average=self.average)
        f1   = f1_score       (y_true, y_pred, average=self.average)

        print(f"Precision: {prec:.3f}")
        print(f"Recall:    {rec:.3f}")
        print(f"F1-score:  {f1:.3f}\n")

        print("Classification Report:\n", classification_report(y_true, y_pred))

        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                      display_labels=self.display_labels)
        fig, ax = plt.subplots(figsize=(5, 5))
        disp.plot(ax=ax, cmap='Blues')
        ax.set_title("Confusion Matrix")
        plt.tight_layout()
        plt.show()

    def plot_feature_importance(self, feature_names, top_n=30):
        if hasattr(self.model, "feature_importances_"):
            # Tree-based models
            importances = self.model.feature_importances_
            indices = importances.argsort()[::-1][:top_n]
    
            plt.figure(figsize=(10, 6))
            plt.barh(range(top_n), importances[indices][::-1])
            plt.yticks(range(top_n), [feature_names[i] for i in indices][::-1])
            plt.title("Top Feature Importances (Tree-based)")
            plt.tight_layout()
            plt.show()
    
        elif hasattr(self.model, "coef_"):
            # Linear models like Logistic Regression
            coefs = self.model.coef_.ravel()
            abs_coefs = np.abs(coefs)
            indices = abs_coefs.argsort()[::-1][:top_n]
    
            plt.figure(figsize=(10, 6))
            plt.barh(range(top_n), coefs[indices][::-1])
            plt.yticks(range(top_n), [feature_names[i] for i in indices][::-1])
            plt.title("Top Logistic Regression Coefficients")
            plt.axvline(0, color='black', linestyle='--', linewidth=0.7)
            plt.tight_layout()
            plt.show()
    
        else:
            print("This model does not support feature importance or coefficients.")



    def fit_and_evaluate(self, X_train, y_train, X_val, y_val):
        print("Training model:")
        self.model.fit(X_train, y_train)

        print("Training Performance:")
        y_train_pred = self.model.predict(X_train)
        self.evaluate_classification(y_train, y_train_pred)

        print("Validation Performance:")
        y_val_pred = self.model.predict(X_val)
        self.evaluate_classification(y_val, y_val_pred)

        return self.model


## Logistic Regression

In [ ]:
# initialize logistic regression
logreg = LogisticRegression(solver='saga', penalty='l1', class_weight='balanced', random_state=42, max_iter=5000)

False Positive Analysis

In [ ]:
def make_false_positive_df(X_validation_df, y_val, model, asin_file_path):
    """
    Creates a DataFrame of false positives (predicted 1 but true label is 0)
    from a classification model's validation predictions, merged with ASIN data.

    Parameters:
        X_validation_df (pd.DataFrame): Validation feature set.
        y_val (pd.Series): True labels with index corresponding to ASINs.
        model (sklearn-like model): Trained model with a `.predict()` method.
        asin_file_path (str): Path to a Parquet file containing ASIN metadata, indexed by ASIN.

    Returns:
        pd.DataFrame: False positive samples merged with ASIN metadata.
    """
    # Predict labels
    preds=model.fit(X_train_final_df,y_train)
    preds = model.predict(X_validation_df)
    probs = model.predict_proba(X_validation_df)[:,1]

    # Create DataFrame with predictions and true labels
    fp_df = X_validation_df.copy()
    fp_df['preds'] = preds
    fp_df['proba'] = probs
    fp_df['original_index'] = y_val.index
    fp_df['y_true'] = y_val.values

    # Load ASIN metadata
    asin_df = pd.read_parquet(asin_file_path)
    asin_df = asin_df.copy()
    asin_df['index'] = asin_df.index  # Preserve index for merging

    # Merge using the original indices (which should align with asin_df's index)
    fp_df = fp_df.merge(asin_df, left_on='original_index', right_on='index')

    # Filter to false positives only: true label 0, predicted 1
    false_positives = fp_df[(fp_df['y_true'] == 0) & (fp_df['preds'] == 1)]

    return false_positives, fp_df



In [ ]:
false_positives, full_val_df = make_false_positive_df(X_val_final_df, y_val, logreg, '/Users/rebekaheichberg/Desktop/git-re/summer-2025-safeify/Data/asins_in_splits/validationA_asins.parquet')

In [ ]:
# check review text to see if negative words appear
full_val_df.sample(10)

In [ ]:
# check review text of true negatives to see how negative words show up in the reviews
full_val_df[full_val_df['y_true']==0].sample(10)

In [ ]:
def false_positive_category_concentration(false_positive_df):
    dummies = list(false_positive_df.columns[:-2])
    un_dummied = false_positive_df[dummies].idxmax(axis=1)
    un_dummied = un_dummied.str.replace('cat_', '', regex=False)
    return un_dummied.value_counts(normalize=True)
    

In [ ]:
def false_positive_category_concentration(false_positive_df, dummy_prefix='category_'):
    """
    Calculates the normalized concentration of false positives across categories.

    Parameters:
        false_positive_df (pd.DataFrame): DataFrame of false positives with one-hot encoded category columns.
        dummy_prefix (str): Prefix used in the one-hot encoded category column names (default is 'cat_').

    Returns:
        pd.Series: Normalized value counts (proportions) for each original category.
    """
    # Automatically detect one-hot encoded category columns
    category_cols = [col for col in false_positive_df.columns if col.startswith(dummy_prefix)]

    if not category_cols:
        raise ValueError(f"No columns found with prefix '{dummy_prefix}'.")

    # Get the most likely category per row (based on the dummy with max value)
    top_category = false_positive_df[category_cols].idxmax(axis=1)

    # Remove prefix to get original category names
    top_category = top_category.str.replace(dummy_prefix, '', regex=False)

    # Return normalized frequency
    return top_category.value_counts(normalize=True)


In [ ]:
false_positive_category_concentration(false_positives)

In [ ]:
# Get normalized counts for each category in false positives
fp_cat_dist = false_positive_category_concentration(false_positives)

# Do the same for true negatives (TN): y_true = 0, preds = 0
tn_df = full_val_df[(full_val_df['y_true'] == 0) & (full_val_df['preds'] == 0)]
tn_cat_dist = false_positive_category_concentration(tn_df)

# Combine into a DataFrame for plotting
comparison_df = pd.DataFrame({
    'False Positives': fp_cat_dist,
    'True Negatives': tn_cat_dist
}).fillna(0)

# Plot
comparison_df.plot(kind='bar', figsize=(12,6), title='Category Distribution: False Positives vs True Negatives')


In [ ]:
from scipy.stats import chi2_contingency

def compute_category_chi2(fp_df, tn_df, dummy_prefix='cat_'):
    # Step 1: Get category columns
    cat_cols = [col for col in fp_df.columns if col.startswith(dummy_prefix)]
    
    # Step 2: Convert back to category labels
    def get_cat_dist(df):
        cat_labels = df[cat_cols].idxmax(axis=1).str.replace(dummy_prefix, '', regex=False)
        return cat_labels.value_counts()
    
    fp_counts = get_cat_dist(fp_df)
    tn_counts = get_cat_dist(tn_df)

    # Step 3: Align index and fill missing categories with 0
    aligned = pd.concat([fp_counts, tn_counts], axis=1, keys=['FP', 'TN']).fillna(0)

    # Step 4: Ensure integer values
    aligned = aligned.astype(int)

    # Step 5: Create contingency table
    contingency = aligned.T.values

    # Step 6: Perform Chi-squared test
    chi2, p, _, _ = chi2_contingency(contingency)

    return p, aligned


In [ ]:
p, cat_table = compute_category_chi2(false_positives, tn_df, dummy_prefix='category_')

In [ ]:
p

* Model is misclassifying certain categories as positive, even though they?re actually negative.
* Some categories are overrepresented in the false positives relative to the true negatives.
* This might indicate:

  * **Feature bias**: Certain features correlate with these categories and are driving the wrong predictions.
  * **Label issues**: These categories may be ambiguous or inconsistently labeled.
  * **Insufficient representation** in training data for those categories.

In [ ]:
cat_table

In [ ]:
cat_table['FP_Rate'] = cat_table['FP'] / cat_table['FP'].sum()
cat_table['TN_Rate'] = cat_table['TN'] / cat_table['TN'].sum()
cat_table['Overrepresented_in_FP'] = cat_table['FP_Rate'] - cat_table['TN_Rate']
cat_table.sort_values('Overrepresented_in_FP', ascending=False).head(10)


False positives with high predicted probabilities could indicate:

- Confusing features
- Potential label noise
- Overconfident model predictions

In [ ]:
# Histogram of probabilities
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(false_positives['proba'], bins=20, kde=True)
plt.title("Predicted Probabilities for False Positives")
plt.xlabel("Predicted Probability")
plt.ylabel("Count")
